In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error, r2_score)

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_csv("D:\\MLOPS_day1\\data\\data.csv")

df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [3]:
X = df[["TV", "radio", "newspaper"]]
y = df["sales"]

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=56)

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [5]:
mlflow.set_experiment("Advertising Sales Prediction")

2026/09/07 17:39:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/07 17:39:53 INFO mlflow.store.db.utils: Updating database tables
2026/09/07 17:39:58 INFO mlflow.tracking.fluent: Experiment with name 'Advertising Sales Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:d:/MLOPS_day1/notebooks/mlruns/1', creation_time=1788782998696, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788782998696, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

In [6]:
with mlflow.start_run(run_name="Linear Regression"):

    model = LinearRegression()

    model.fit(Xtrain, ytrain)

    y_pred = model.predict(Xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

In [7]:
with mlflow.start_run(run_name="Ridge Regression"):

    model = Ridge(alpha=1.0)

    model.fit(Xtrain, ytrain)

    y_pred = model.predict(Xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    # artifacts
    mlflow.sklearn.log_model(sk_model = model , name = "Ridge Regression Model")

In [8]:
mlflow.sklearn.autolog()

In [9]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:

    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    model.fit(Xtrain, ytrain)

    test_pred = model.predict(Xtest)


    test_rmse = root_mean_squared_error(ytest, test_pred)
    test_mae = mean_absolute_error(ytest, test_pred)
    test_r2 = r2_score(ytest, test_pred)

    # Custom project metrics
    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

2026/09/07 17:56:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [10]:
run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/2288666bf78e44dc8efbdf82a25101bf/model


In [11]:
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="Advertising_Sales_model"
)


Successfully registered model 'Advertising_Sales_model'.
2026/09/07 18:01:21 WARNING mlflow.tracking._model_registry.fluent: Run with id 2288666bf78e44dc8efbdf82a25101bf has no artifacts at artifact path 'model', registering model based on models:/m-b125ef0e52334eda89868805e7665a0b instead
Created version '1' of model 'Advertising_Sales_model'.


In [12]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(

    name="Advertising_Sales_model",
    alias="champian",
    version ="1"
)

In [13]:
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_model@champian"
)

new_data = pd.DataFrame({
    "TV": [150, 200],
    "radio": [30, 40],
    "newspaper": [20, 30]
})

print(model.predict(new_data))

[16.20536288 19.92067682]
